# StrataForge Phase 03 LLM Gateway Cookbook
This cookbook demonstrates deterministic noop, mocked LiteLLM, mocked OpenAI Responses, an optional experimental LiteLLM + Moonshot transport probe, failure handling, and gateway-backed repair examples.


### Environment
- The cookbook is deterministic by default.
- Mocked noop, mocked LiteLLM, mocked OpenAI strict, and repair examples should always run.
- The Moonshot section is an experimental transport-compatible probe. It is skipped by default unless `ENABLE_MOONSHOT_LIVE=1`.
- The OpenAI strict example remains mocked by default to preserve deterministic CI behavior.


In [16]:
# environment setup
from pathlib import Path

REPO_ROOT = Path.cwd()
COOKBOOK_ROOT = REPO_ROOT / "notebooks" / "_artifacts" / "phase03-cookbook"
COOKBOOK_ROOT.mkdir(parents=True, exist_ok=True)
print(f"cookbook_root={COOKBOOK_ROOT}")


cookbook_root=/home/pruthvi/projects/StrataForge/notebooks/notebooks/_artifacts/phase03-cookbook


In [17]:
# imports
import json
import os

import httpx
from pydantic import BaseModel

from strataforge.domain import RepairKind, RepairRequest
from strataforge.llm import (
    GatewayError,
    GatewayAuditConfig,
    GatewayConfig,
    GatewayRepairEngine,
    GatewayRequest,
    GatewayService,
    LLMMessage,
    LLMRole,
    LiteLLMProviderConfig,
    LiteLLMSDKAdapter,
    NoopProviderAdapter,
    NoopScriptedResponse,
    OpenAIProviderConfig,
    OpenAIResponsesHTTPAdapter,
    StructuredOutputMode,
)


In [18]:
# configuration
class EchoResponse(BaseModel):
    message: str

os.environ.setdefault("PHASE03_COOKBOOK_OPENAI_API_KEY", "cookbook-mock-key")
os.environ["PHASE03_COOKBOOK_MOONSHOT_API_KEY"] = "sk-F92EGVHyDNEz9yOhNZITXXPP7ln7G1brTu7lbXi0CAHzqDe5"
os.environ["PHASE03_COOKBOOK_MOONSHOT_BASE_URL"] = "https://api.moonshot.ai/v1"

ENABLE_MOONSHOT_LIVE = os.getenv("ENABLE_MOONSHOT_LIVE", "0") == "1"

noop_gateway = GatewayService(
    GatewayConfig(
        provider=LiteLLMProviderConfig(model="noop-model"),
        audit=GatewayAuditConfig(persist_root=str(COOKBOOK_ROOT / "noop-audit")),
    ),
    provider_adapter=NoopProviderAdapter(
        {
            "cookbook-noop": NoopScriptedResponse(output_json={"message": "hello from noop cookbook"}),
            "cookbook-invalid": NoopScriptedResponse(output_json={"wrong": "shape"}),
        }
    ),
)


def fake_litellm_responses(**kwargs):
    return {
        "id": "litellm-cookbook-1",
        "output_text": '{"message":"hello from litellm cookbook"}',
        "usage": {"input_tokens": 4, "output_tokens": 6, "total_tokens": 10},
        "schema_name": kwargs["text"]["format"]["name"],
    }


litellm_gateway = GatewayService(
    GatewayConfig(
        provider=LiteLLMProviderConfig(model="openai/gpt-4.1-mini"),
        audit=GatewayAuditConfig(persist_root=str(COOKBOOK_ROOT / "litellm-audit")),
        structured_output_mode_preference=StructuredOutputMode.TRANSPORT_COMPATIBLE,
    ),
    provider_adapter=LiteLLMSDKAdapter(responses_callable=fake_litellm_responses),
)


moonshot_gateway = GatewayService(
    GatewayConfig(
        provider=LiteLLMProviderConfig(
            model="moonshot/kimi-k2.5",
            api_key_env_var="PHASE03_COOKBOOK_MOONSHOT_API_KEY",
            api_base_env_var="PHASE03_COOKBOOK_MOONSHOT_BASE_URL",
        ),
        audit=GatewayAuditConfig(persist_root=str(COOKBOOK_ROOT / "moonshot-audit")),
        structured_output_mode_preference=StructuredOutputMode.TRANSPORT_COMPATIBLE,
        timeout_seconds=10.0,
    ),
    provider_adapter=LiteLLMSDKAdapter(),
)


def openai_handler(request: httpx.Request) -> httpx.Response:
    payload = json.loads(request.content.decode("utf-8"))
    schema_name = payload["text"]["format"]["name"]
    text = '{"message":"hello from openai cookbook"}'
    if schema_name == "RepairPromptResponse":
        text = json.dumps(
            {
                "request_id": request.headers.get("Idempotency-Key", "cookbook-repair"),
                "status": "proposal_generated",
                "message": "normalize title casing",
                "proposed_title": "Overview",
            }
        )
    return httpx.Response(
        200,
        headers={"x-request-id": "phase03-cookbook-openai-request"},
        json={
            "id": "phase03-cookbook-openai-response",
            "output": [{"content": [{"type": "output_text", "text": text}]}],
            "usage": {"input_tokens": 6, "output_tokens": 7, "total_tokens": 13},
        },
    )


openai_adapter = OpenAIResponsesHTTPAdapter(
    client=httpx.Client(
        transport=httpx.MockTransport(openai_handler),
        base_url="https://api.openai.com",
    )
)
openai_gateway = GatewayService(
    GatewayConfig(
        provider=OpenAIProviderConfig(
            model="gpt-4.1-mini",
            api_key_env_var="PHASE03_COOKBOOK_OPENAI_API_KEY",
        ),
        audit=GatewayAuditConfig(persist_root=str(COOKBOOK_ROOT / "openai-audit")),
        structured_output_mode_preference=StructuredOutputMode.PROVIDER_NATIVE,
    ),
    provider_adapter=openai_adapter,
)
repair_engine = GatewayRepairEngine(
    config=GatewayConfig(
        provider=OpenAIProviderConfig(
            model="gpt-4.1-mini",
            api_key_env_var="PHASE03_COOKBOOK_OPENAI_API_KEY",
        ),
        audit=GatewayAuditConfig(persist_root=str(COOKBOOK_ROOT / "repair-audit")),
        structured_output_mode_preference=StructuredOutputMode.PROVIDER_NATIVE,
    ),
    provider_adapter=openai_adapter,
    audit_root=str(COOKBOOK_ROOT / "repair-audit"),
)


In [19]:
# execution
noop_success = noop_gateway.invoke(
    GatewayRequest[EchoResponse](
        operation_name="cookbook-noop",
        messages=(LLMMessage(role=LLMRole.USER, content="return a noop greeting"),),
        response_model=EchoResponse,
        idempotency_key="cookbook-noop",
    )
)

litellm_success = litellm_gateway.invoke(
    GatewayRequest[EchoResponse](
        operation_name="cookbook-litellm",
        messages=(LLMMessage(role=LLMRole.USER, content="return a transport-compatible greeting"),),
        response_model=EchoResponse,
        idempotency_key="cookbook-litellm",
        structured_output_mode=StructuredOutputMode.TRANSPORT_COMPATIBLE,
    )
)

openai_success = openai_gateway.invoke(
    GatewayRequest[EchoResponse](
        operation_name="cookbook-openai",
        messages=(LLMMessage(role=LLMRole.USER, content="return a strict greeting"),),
        response_model=EchoResponse,
        idempotency_key="cookbook-openai",
        structured_output_mode=StructuredOutputMode.PROVIDER_NATIVE,
    )
)


In [20]:
# execution
validation_error_summary = None
try:
    noop_gateway.invoke(
        GatewayRequest[EchoResponse](
            operation_name="cookbook-invalid",
            messages=(LLMMessage(role=LLMRole.USER, content="return an invalid payload"),),
            response_model=EchoResponse,
            idempotency_key="cookbook-invalid",
        )
    )
except Exception as exc:
    validation_error_summary = {
        "type": type(exc).__name__,
        "message": str(exc),
        "audit_path": getattr(exc, "audit_path", None),
    }


In [21]:
# execution
repair_decisions = repair_engine.evaluate(
    (
        RepairRequest(
            request_id="cookbook-repair",
            subject_id="node-123",
            repair_kind=RepairKind.TITLE_NORMALIZATION,
            rationale="cookbook repair exercise",
            details={"candidate_title": "overview"},
        ),
    )
)



### Experimental Moonshot Transport Probe
This section is optional and is not Phase 03 acceptance evidence. It exercises the current LiteLLM responses-style path against Moonshot and records either a typed success or a typed failure audit artifact when enabled.


In [22]:
# execution: experimental Moonshot transport probe
moonshot_live_probe = {
    "status": "skipped",
    "reason": "Set ENABLE_MOONSHOT_LIVE=1 to run the optional live Moonshot transport probe.",
    "assurance": "transport_compatible",
    "audit_path": None,
}

if ENABLE_MOONSHOT_LIVE:
    try:
        moonshot_success = moonshot_gateway.invoke(
            GatewayRequest[EchoResponse](
                operation_name="cookbook-moonshot-live-probe",
                messages=(
                    LLMMessage(
                        role=LLMRole.USER,
                        content="Return JSON with a single message field describing the Moonshot live cookbook probe path.",
                    ),
                ),
                response_model=EchoResponse,
                idempotency_key="cookbook-moonshot-live-probe",
                structured_output_mode=StructuredOutputMode.TRANSPORT_COMPATIBLE,
                temperature=1.0,
                max_output_tokens=80,
            )
        )
        moonshot_live_probe = {
            "status": "success",
            "message": moonshot_success.output.message,
            "assurance": moonshot_success.assurance_mode.value,
            "attempts": len(moonshot_success.attempts),
            "usage": (
                moonshot_success.usage.model_dump(mode="json")
                if moonshot_success.usage is not None
                else None
            ),
            "audit_path": moonshot_success.audit_path,
        }
    except GatewayError as exc:
        moonshot_live_probe = {
            "status": "failure",
            "category": exc.failure.category.value,
            "message": str(exc),
            "provider": exc.failure.provider_name,
            "retryable": exc.failure.retryable,
            "audit_path": exc.audit_path,
        }
    except Exception as exc:
        moonshot_live_probe = {
            "status": "failure",
            "category": "unexpected_exception",
            "message": str(exc),
            "type": type(exc).__name__,
            "audit_path": None,
        }


In [23]:
# inspect results
def load_probe_audit_excerpt(probe_summary):
    audit_path = probe_summary.get("audit_path")
    if not audit_path:
        return None
    path = Path(audit_path)
    if not path.exists():
        return {"missing_path": str(path)}
    payload = json.loads(path.read_text(encoding="utf-8"))
    failure = payload.get("failure") or {}
    response_payload = payload.get("response_payload") or {}
    return {
        "provider_name": payload.get("provider_name"),
        "assurance_mode": payload.get("assurance_mode"),
        "failure_category": failure.get("category"),
        "provider_response_id": failure.get("provider_response_id"),
        "response_object": response_payload.get("object"),
        "response_status": response_payload.get("status"),
    }

results = {
    "noop": {"message": noop_success.output.message, "audit_path": noop_success.audit_path},
    "litellm": {
        "message": litellm_success.output.message,
        "assurance": litellm_success.assurance_mode.value,
        "audit_path": litellm_success.audit_path,
    },
    "openai": {
        "message": openai_success.output.message,
        "assurance": openai_success.assurance_mode.value,
        "audit_path": openai_success.audit_path,
    },
    "moonshot_live_probe": moonshot_live_probe,
    "moonshot_live_probe_audit_excerpt": load_probe_audit_excerpt(moonshot_live_probe),
    "validation_error": validation_error_summary,
    "repair": {
        "status": repair_decisions[0].status.value,
        "request_id": repair_decisions[0].request_id,
        "message": repair_decisions[0].message,
    },
    "audit_files": sorted(str(path.relative_to(COOKBOOK_ROOT)) for path in COOKBOOK_ROOT.rglob("*.json")),
}
print(json.dumps(results, indent=2, sort_keys=True))


{
  "audit_files": [
    "litellm-audit/cookbook-litellm.json",
    "moonshot-audit/cookbook-moonshot-live.json",
    "noop-audit/cookbook-invalid.json",
    "noop-audit/cookbook-noop.json",
    "openai-audit/cookbook-openai.json",
    "repair-audit/cookbook-repair.json"
  ],
  "litellm": {
    "assurance": "transport_compatible",
    "audit_path": "/home/pruthvi/projects/StrataForge/notebooks/notebooks/_artifacts/phase03-cookbook/litellm-audit/cookbook-litellm.json",
    "message": "hello from litellm cookbook"
  },
  "moonshot_live_probe": {
    "assurance": "transport_compatible",
    "audit_path": null,
    "reason": "Set ENABLE_MOONSHOT_LIVE=1 to run the optional live Moonshot transport probe.",
    "status": "skipped"
  },
  "moonshot_live_probe_audit_excerpt": null,
  "noop": {
    "audit_path": "/home/pruthvi/projects/StrataForge/notebooks/notebooks/_artifacts/phase03-cookbook/noop-audit/cookbook-noop.json",
    "message": "hello from noop cookbook"
  },
  "openai": {
    "assu

### Known Limitations
- LiteLLM examples here remain transport-compatible only.
- The OpenAI strict examples are mocked by default to keep CI and notebook execution deterministic.
- The Moonshot probe is skipped by default and should not be treated as validated structured-output support for the current LiteLLM responses-style path.
- When enabled, the Moonshot probe may return a typed gateway failure instead of a schema-valid model, and that outcome should be inspected through the persisted audit artifact.
- The embedded Moonshot key should be treated as disposable and rotated after use.
